# Experiments: Testset Predicts

In [1]:
import sys
import os
project_root = "" # set the root path
sys.path.insert(0, project_root)

import h5py
import torch
from tensorboard.notebook import display
from functions.helper import batch_wise_random_drop, obs_field_reconstruction, sampler_test_data_generator, sampler_checkpoint_load, sampler_training_visualizer
from functions.data import dataloader
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
device = [i for i in range(torch.cuda.device_count())][0]

In [2]:
secnario = "SHM" 

In [3]:
PIS_dir = "../output/"
obs_sample = [4096, 256, 64, 32, 16, 12]
PIS_dir_list = os.listdir(PIS_dir)

In [4]:
test_loader, _ = dataloader(
    data_dir = f"../data/{secnario}/testset.h5",
    shuffle = False,
    batch_size = 32,
    num_workers = 4,)

print(f"Dataloader: <{secnario}> completed.")

PIS = {}

for i in PIS_dir_list:
    if secnario in i:
        pis_dir = "../output/"+i
        break

for obs_count in obs_sample:
    if obs_count == 4096:
        model, certainty_param = sampler_checkpoint_load(pis_dir, 1, obs_count, "closest", device)
    else:
        model, certainty_param = sampler_checkpoint_load(pis_dir, 2, obs_count, "closest", device)
    PIS[str(obs_count)] = {"model":model, "certainty_param":certainty_param}

print(f"PIS: <{PIS.keys()}> completed.")

Dataloader: <SHM> completed.
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_1_best.pt
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_2_256.pt
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_2_64.pt
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_2_32.pt
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_2_16.pt
-> Field Inversion Checkpoint loaded: output/SHM-2025-12-28-04-58-41/checkpoints/stage_2_12.pt
PIS: <dict_keys(['4096', '256', '64', '32', '16', '12'])> completed.


In [5]:
pred_dicts = {}

for sp in obs_sample:
    output_list = []
    for inputs in tqdm(test_loader):
        target = inputs["target"].to(device)
        obs = inputs["obs"].to(device)
        drop_obs = batch_wise_random_drop(obs, sp)
        
        output = PIS[str(sp)]["model"].sample(shape=target.shape, steps=80, device="cuda", conditions=drop_obs)     
        output_list.append(output)

    output_result = torch.cat(output_list, dim = 0).squeeze(1)
    pred_dicts[str(sp)] = output_result

100%|██████████| 7/7 [00:11<00:00,  1.68s/it]


In [6]:
with h5py.File(f"Outputs/PIS_Test_Predicts_{secnario}.h5", "w") as f:
        for key_val in pred_dicts.keys():
            f.create_dataset(key_val, data = pred_dicts[key_val].detach().cpu().numpy())